In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import make_column_selector,make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [14]:
df = pd.read_csv("../Datasets/loan.csv",index_col=0)
df

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
Loan_ID,,,,,,,,,,,,
LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...
LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y


In [15]:
df.isna().sum()

Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [16]:
df.isna().sum().sum()

149

In [17]:
X, y = df.drop("Loan_Status", axis=1), df["Loan_Status"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3,stratify=y)

In [20]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
imp_cat = SimpleImputer(strategy="most_frequent").set_output(transform='pandas')
imp_num = SimpleImputer().set_output(transform='pandas')
ct_imp = make_column_transformer((imp_cat,make_column_selector(dtype_include=object)),
                                 (imp_num,make_column_selector(dtype_include=['int64','float64'])),
                                 verbose_feature_names_out=False)
ct_imp = ct_imp.set_output(transform='pandas')
X_trns = ct_imp.fit_transform(X)
X_trns.isna().sum().sum()

0

In [25]:
ct_imp

ColumnTransformer(transformers=[('simpleimputer-1',
                                 SimpleImputer(strategy='most_frequent'),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x00000123E9978170>),
                                ('simpleimputer-2', SimpleImputer(),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x00000123E99794F0>)],
                  verbose_feature_names_out=False)

In [21]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct_ohe = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct_ohe = ct_ohe.set_output(transform='pandas')

In [24]:
scaler = StandardScaler().set_output(transform='pandas')
ks = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
scores = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    pipe = Pipeline([("ct_imp",ct_imp),("ct_ohe",ct_ohe),("Scaler",scaler),("knn",knn)])
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    scores.append([k,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores, columns=["K","Score"])
df_scores.sort_values("Score",ascending=False)

,K,Score
5,6,0.843243
6,7,0.843243
7,8,0.843243
8,9,0.837838
9,10,0.837838
10,11,0.837838
12,13,0.837838
13,14,0.837838
4,5,0.832432
11,12,0.832432


In [26]:
scaler = MinMaxScaler().set_output(transform='pandas')
ks = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
scores = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    pipe = Pipeline([("ct_imp",ct_imp),("ct_ohe",ct_ohe),("Scaler",scaler),("knn",knn)])
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    scores.append([k,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores, columns=["K","Score"])
df_scores.sort_values("Score",ascending=False)

,K,Score
4,5,0.821622
5,6,0.821622
7,8,0.821622
6,7,0.810811
2,3,0.805405
9,10,0.789189
11,12,0.789189
13,14,0.789189
3,4,0.778378
10,11,0.778378
